## 1. Importación de librerías y carga de datos

En esta primera sección preparamos nuestro entorno de trabajo cargando las librerías necesarias para el procesamiento de texto y NLP. A continuación, importamos el archivo CSV de comentarios que generamos en la fase de estructuración anterior.

In [ ]:
# Librerías necesarias para NLP y preprocesado
import pandas as pd
import re
import emoji # Recuerda hacer !pip install emoji si no lo tienes instalado en este entorno

# Cargar el dataset de comentarios
ruta_comentarios = "datos/procesados/df_hoteles_vlc_comentarios.csv"
df_comentarios = pd.read_csv(ruta_comentarios)

print(f"Total de comentarios cargados: {len(df_comentarios)}")
display(df_comentarios.head(3))

## 2. Traducción y tratamiento de Emojis

Los emojis contienen información sentimental muy valiosa para modelos de NLP, pero muchos algoritmos no saben interpretarlos directamente. En esta celda definimos una función que detecta los emojis en el texto y los traduce a su significado en texto plano en español (ej: 😍 pasa a ser "cara sonriendo con ojos de corazón").

In [ ]:
# Función para transformar emojis a texto descriptivo en español
def handle_emoji(string):
    if not isinstance(string, str):
        return string
        
    # Buscamos los emojis únicos presentes en la cadena
    emojis_encontrados = set(c['emoji'] for c in emoji.emoji_list(string))
    
    for emj in emojis_encontrados:
        # demojize lo convierte a español con formato ":cara_sonriendo:"
        nombre_es = emoji.demojize(emj, language='es')
        
        # Limpiamos el texto: quitamos los dos puntos y cambiamos guiones bajos por espacios
        texto_limpio = nombre_es.replace(':', '').replace('_', ' ')
        
        # Reemplazamos el emoji por su texto
        string = string.replace(emj, f" {texto_limpio} ")
        
    # Eliminar posibles espacios dobles generados
    return " ".join(string.split())

# Seleccionamos solo las columnas que son de texto para no afectar a las fechas o números
string_cols = df_comentarios.select_dtypes(include=['object', 'string']).columns

# Aplicamos la función a todas las columnas de texto
df_comentarios[string_cols] = df_comentarios[string_cols].apply(lambda col: col.apply(handle_emoji))

print("✓ Emojis transformados a texto descriptivo con éxito.")

## 3. Normalización: Minúsculas, enlaces y caracteres especiales

Para homogeneizar el texto y reducir el ruido en nuestro futuro modelo, pasamos todos los caracteres a minúsculas. Además, mediante expresiones regulares (Regex), eliminamos cualquier enlace web (URLs) y limpiamos los caracteres extraños, conservando únicamente caracteres alfanuméricos y signos de puntuación básicos.

In [ ]:
# 1. Pasar todo a minúsculas
df_comentarios[string_cols] = df_comentarios[string_cols].apply(lambda col: col.str.lower())

# 2. Eliminar enlaces (URLs) y quitar espacios extra a los lados
df_comentarios[string_cols] = df_comentarios[string_cols].apply(
    lambda col: col.str.replace(r'https?://\S+|www\.\S+', '', regex=True).str.strip()
)

# 3. Eliminar caracteres especiales no deseados
# Nos quedamos con minúsculas (incluyendo tildes y ñ), números, espacios y puntuación básica
df_comentarios[string_cols] = df_comentarios[string_cols].apply(
    lambda col: col.str.replace(r'[^a-záéíóúüñàèìòùâêîôûäëïöüa-z0-9\s\[\].,!?;:\-]', ' ', regex=True)
)

print("✓ Texto normalizado (minúsculas, sin enlaces y sin caracteres especiales).")

## 4. Guardado del Dataset Preprocesado

Una vez finalizada la limpieza y normalización del texto, exportamos el DataFrame resultante a un nuevo archivo CSV. 

In [ ]:
# Exportar el dataset final preparado para NLP
ruta_salida = "datos/procesados/df_comentarios_limpio.csv"
df_comentarios.to_csv(ruta_salida, index=False, encoding='utf-8-sig')

print(f"¡Archivo '{ruta_salida}' creado con éxito!")

# Verificamos visualmente las columnas de texto principales
display(df_comentarios[['comentario_general', 'positivo', 'negativo']].head(5))